# Business Intelligence Analysis — Clean Shipments

This notebook executes key SQL queries against the `clean_shipments` table to derive actionable business insights on shipping operations, costs, and customer behaviour.

## Import Libraries

In [4]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(r"../data/logistics_clean.db")
print("Connected to database successfully.")

Connected to database successfully.


## Query 1: Total Shipments by Shipping Mode

**Business Purpose:** Understand which shipping modes are most frequently used.

**Expected Insight:** Identifies the dominant shipping mode so logistics strategy can be optimised.

In [5]:
query = """
SELECT shipping_mode,
       COUNT(*) AS total_shipments
FROM clean_shipments
GROUP BY shipping_mode
ORDER BY total_shipments DESC;
"""
result = pd.read_sql_query(query, conn)
display(result)

,shipping_mode,total_shipments
0,air,62511
1,trk,42374
2,sea,42343
3,Air Freight,42302
4,AIR,42242
5,TRK,42168
6,rail,42091
7,Sea,42086
8,truck,41883


**Interpretation:** The shipping mode with the highest count represents the workhorse of the logistics network. Resources and cost-reduction efforts should be prioritised there.

## Query 2: Average Shipment Cost by Shipping Mode

**Business Purpose:** Compare cost efficiency across shipping modes.

**Expected Insight:** Reveals which modes are most/least cost-effective on average.

In [6]:
query = """
SELECT shipping_mode,
       ROUND(AVG(total_cost_usd), 2) AS avg_cost_usd
FROM clean_shipments
GROUP BY shipping_mode
ORDER BY avg_cost_usd DESC;
"""
result = pd.read_sql_query(query, conn)
display(result)

,shipping_mode,avg_cost_usd
0,Air Freight,1707.35
1,rail,1704.16
2,TRK,1703.54
3,AIR,1703.32
4,truck,1702.18
5,trk,1702.18
6,air,1701.34
7,Sea,1701.12
8,sea,1698.57


**Interpretation:** Modes with high average cost may justify negotiation for better rates or a shift to cheaper alternatives where service levels allow.

## Query 3: Top 10 Destination Countries

**Business Purpose:** Identify the most common destination countries.

**Expected Insight:** Highlights key markets that drive shipment volume.

In [7]:
query = """
SELECT destination_country,
       COUNT(*) AS shipment_count
FROM clean_shipments
GROUP BY destination_country
ORDER BY shipment_count DESC
LIMIT 10;
"""
result = pd.read_sql_query(query, conn)
display(result)

,destination_country,shipment_count
0,KHM,104689
1,Thailand,42269
2,Vietnam,42240
3,TH,42204
4,Cambodia,42183
5,VN,42158
6,VNM,42136
7,KH,42121


**Interpretation:** The top destination countries should be the focus of regional partnerships, hub investments, and service-level agreements.

## Query 4: Most Frequent Shipment Status

**Business Purpose:** Understand the overall health of the shipment pipeline.

**Expected Insight:** A high percentage of "Delivered" status indicates operational health; a large share of "In Transit" or "Pending" may flag bottlenecks.

In [8]:
query = """
SELECT status,
       COUNT(*) AS cnt,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM clean_shipments), 2) AS pct_of_total
FROM clean_shipments
GROUP BY status
ORDER BY cnt DESC;
"""
result = pd.read_sql_query(query, conn)
display(result)

,status,cnt,pct_of_total
0,in transit,50791,12.70
1,INTRANSIT,38935,9.73
2,lost,38902,9.73
3,delivered,38888,9.72
4,LOST,38842,9.71
5,DELIVERED,38802,9.70
6,??,38775,9.69
7,returned,38730,9.68
8,RETURNED,38713,9.68
9,Delivered,38622,9.66


**Interpretation:** If the "Delivered" share is below expectations, the business should investigate delays in the remaining status categories.

## Query 5: Monthly Shipment Volume

**Business Purpose:** Detect seasonality and growth trends.

**Expected Insight:** Peaks and troughs throughout the year inform capacity planning and promotional timing.

In [9]:
query = """
SELECT strftime('%Y-%m', ship_date) AS year_month,
       COUNT(*) AS shipment_count
FROM clean_shipments
WHERE ship_date IS NOT NULL
GROUP BY year_month
ORDER BY year_month;
"""
result = pd.read_sql_query(query, conn)
display(result)

,year_month,shipment_count
0,2020-01,5574
1,2020-02,5105
2,2020-03,5408
3,2020-04,5329
4,2020-05,5481
...,...,...
67,2025-08,5456
68,2025-09,5307
69,2025-10,5433
70,2025-11,5318


**Interpretation:** Monthly volume patterns help schedule maintenance, staffing, and inventory ahead of peak periods.

## Query 6: Average Delivery Duration

**Business Purpose:** Measure average number of days from ship to delivery.

**Expected Insight:** Baseline transit time for performance monitoring and customer promises.

In [10]:
query = """
SELECT ROUND(AVG(julianday(delivery_date) - julianday(ship_date)), 1) AS avg_delivery_days
FROM clean_shipments
WHERE delivery_date IS NOT NULL
  AND ship_date IS NOT NULL;
"""
result = pd.read_sql_query(query, conn)
display(result)

,avg_delivery_days
0,0.9


**Interpretation:** The average delivery duration sets the benchmark. Any significant deviation in recent periods signals operational issues.

## Query 7: Highest Cost Routes (Origin City → Destination Country)

**Business Purpose:** Identify the most expensive origin–destination pairs.

**Expected Insight:** Enables targeted renegotiation or mode switching for costly routes.

In [11]:
query = """
SELECT origin_city,
       destination_country,
       ROUND(AVG(total_cost_usd), 2) AS avg_cost_usd
FROM clean_shipments
GROUP BY origin_city, destination_country
ORDER BY avg_cost_usd DESC;
"""
result = pd.read_sql_query(query, conn)
display(result)

,origin_city,destination_country,avg_cost_usd
0,PP,TH,1719.65
1,Bangkok,KH,1717.82
2,HCMC,Vietnam,1715.18
3,PP,Vietnam,1714.34
4,Phnompenh,VN,1713.85
...,...,...,...
59,BKK,KHM,1693.54
60,Saigon,KHM,1691.75
61,Saigon,Cambodia,1691.13
62,BKK,TH,1689.92


**Interpretation:** High-cost routes should be reviewed for alternative carriers, consolidation opportunities, or mode shifts.

## Query 8: Customer Shipment Frequency

**Business Purpose:** Profile customer behaviour by shipment count, average cost, and total spend.

**Expected Insight:** Segments customers into high-frequency / high-value tiers for loyalty programmes.

In [12]:
query = """
SELECT customer_name,
       COUNT(*) AS shipment_count,
       ROUND(AVG(total_cost_usd), 2) AS avg_cost_usd,
       ROUND(SUM(total_cost_usd), 2) AS total_cost_usd
FROM clean_shipments
GROUP BY customer_name
ORDER BY total_cost_usd DESC;
"""
result = pd.read_sql_query(query, conn)
display(result)

,customer_name,shipment_count,avg_cost_usd,total_cost_usd
0,Sok Lee,33014,1705.35,56300462.84
1,Srey Nguyen,5125,1704.33,8734703.17
2,Lina Nguyen,5080,1716.37,8719183.41
3,Chan Garcia,5095,1708.56,8705090.04
4,Vann Tran,5099,1706.58,8701867.56
...,...,...,...,...
319,maria smith,52,1702.42,88526.07
320,MARIA LEE,53,1655.91,87763.18
321,LINA KIM,49,1705.75,83581.76
322,CHAN KIM,51,1590.54,81117.38


**Interpretation:** Customers with high frequency and high total spend are prime candidates for volume discounts and dedicated account management.

## Conclusion

The eight queries above provide a comprehensive view of the logistics operation:
- **Mode distribution & cost** guide carrier strategy.
- **Geographic concentration** highlights key markets.
- **Status health** and **delivery duration** flag service issues.
- **Monthly trends** support capacity planning.
- **Route-level & customer-level** insights enable targeted commercial actions.

Together these insights form a foundation for data-driven supply-chain decisions.

In [13]:
conn.close()
print("Database connection closed.")

Database connection closed.
